# V-Energy Grid 360 | Gold — Priorização de perdas não técnicas

**Rocket Lab 2026.2 · Engenharia de Dados · Entregável técnico**  
**Versão:** 4.0 · **Plataforma:** Databricks Serverless / PySpark / Delta Lake  
**Responsabilidade:** construção de uma fila **auditável** de unidades consumidoras (UCs) para avaliação pela equipe de perdas.

## 1. Objetivo e escopo

A distribuidora precisa selecionar as UCs que merecem investigação de possível perda não técnica, considerando que a capacidade de inspeção é limitada. Este notebook produz uma **heurística de priorização** com base em alterações de consumo, qualidade das leituras e fatores que podem explicar a mudança — como geração distribuída (GD) e troca de medidor.

**Entradas, no catálogo informado pelo widget:**

| Tabela Silver | Uso na Gold |
| --- | --- |
| `tb_unidades_consumidoras` | Universo de UCs e dimensões geográficas/comerciais |
| `tb_leituras` | Consumo mensal, tipo de leitura e flags de qualidade |
| `tb_medidores` | Datas de retirada e detecção de trocas de medidor |
| `tb_geracao_distribuida` | Data de conexão da GD |
| `tb_inspecoes` | Inspeções passadas e avaliação retrospectiva opcional |

**Saídas (Delta):** `gold.tb_fila_perdas` — **1 registro por UC elegível** — e `gold.tb_fila_perdas_auditoria` — **1 registro por UC do cadastro**, incluindo excluídas e casos a revisar. Nenhuma tabela Silver ou ordem de inspeção é alterada.

> **Interpretação:** `score` não é probabilidade de fraude. `kwh_estimado` é um *proxy* de redução sazonal de consumo, **não** kWh comprovadamente furtados, recuperáveis ou já recuperados. A decisão de inspeção exige avaliação humana.

**Confidencialidade:** os dados do case são restritos. Antes de enviar ao GitHub, remova as saídas das células, não versionando nomes de titulares, IDs/linhas de UCs, CPFs nem amostras reais.


## 2. Configuração da execução

1. Execute previamente a **Bronze → Silver** no mesmo catálogo; as cinco fontes acima precisam existir com os nomes `silver.tb_*`.
2. Escolha **Serverless** como computação e use `catalog=workspace` (ou o catálogo validado pela equipe).
3. Mantenha `executar_backtest=false` nas execuções operacionais. O backtest é opcional e custa computação adicional.
4. Rode **Run All** e confira as verificações de qualidade antes da publicação Delta.

### Parâmetros de negócio — proposta inicial

- **Janela atual:** últimos 3 meses de competência disponíveis. **Referência sazonal:** os mesmos 3 meses do ano anterior.
- **Histórico mínimo:** 2 pares mensais válidos; pelo menos **1 par com leitura não estimada** nos dois anos.
- **Score elegível:** a partir de 20 pontos; fatores de revisão podem impedir a elegibilidade mesmo acima desse limite.
- **Inspeção recente:** 90 dias; **troca recente de medidor:** 6 meses.
- **Score heurístico:** queda percentual (máx. 55), redução absoluta mensal em kWh (máx. 30), recorrência de leituras estimadas nos pares (10), menos atenuantes. Pesos **não calibrados**.

As regras são versionadas e explícitas. Não modifique limiares, pesos ou contratos da Silver sem alinhamento com os responsáveis das tabelas e do backend.


In [0]:
from datetime import timedelta, datetime, timezone
from pyspark.sql import functions as F

VERSAO_REGRA = "heuristica_v4_sazonal_auditavel"
ID_EXECUCAO = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
JANELA_MESES = 3
MIN_PARES = 2
MIN_PARES_OBSERVADOS = 1
LIMIAR_PROP_PARES_ESTIMADOS = 2.0 / 3.0
LIMIAR_QUEDA_PCT = 0.20
LIMIAR_QUEDA_KWH_MES = 50.0
LIMIAR_ABS_ELEVADO_KWH_MES = 120.0
MIN_SCORE_ELEGIVEL = 20.0
DIAS_INSPECAO_RECENTE = 90
MESES_TROCA_RECENTE = 6
TOP_N = 200
TOPS_BACKTEST = (50, 200)
HORIZONTE_BACKTEST_MESES = 3
PASSO_BACKTEST_MESES = 3
MAX_CORTES_BACKTEST = 4
MIN_ROTULOS_COMPARACAO = 10

# Pesos de triagem NÃO CALIBRADOS; score não representa probabilidade.
PESO_QUEDA_PCT_MAX = 55.0
PESO_QUEDA_KWH_MAX = 30.0
PESO_ESTIMADAS = 10.0
PENALIDADE_GD_APOS_BASE = 35.0
PENALIDADE_TROCA = 30.0
PENALIDADE_INSPECAO = 25.0

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("executar_backtest", "false", ["false", "true"])
CATALOGO = dbutils.widgets.get("catalog").strip()
EXECUTAR_BACKTEST = dbutils.widgets.get("executar_backtest") == "true"
assert CATALOGO.replace("_", "").isalnum(), "Nome de catálogo inválido"
SILVER = f"{CATALOGO}.silver"
GOLD = f"{CATALOGO}.gold"

def ler_silver(nome):
    return spark.table(f"{SILVER}.tb_{nome}")

uc = ler_silver("unidades_consumidoras")
leituras = ler_silver("leituras")
medidores = ler_silver("medidores")
gd = ler_silver("geracao_distribuida")
inspecoes = ler_silver("inspecoes")

for nome, df in [("cadastro", uc), ("leituras", leituras),
                 ("medidores", medidores), ("GD", gd), ("inspecoes", inspecoes)]:
    print(f"{nome}: {', '.join(df.columns)}")
print(f"Regra={VERSAO_REGRA} | execução={ID_EXECUCAO} | backtest={EXECUTAR_BACKTEST}")


## 3. Contratos de entrada e integridade da Silver

O notebook verifica a existência das colunas necessárias e a unicidade do cadastro. O consumo é convertido para análise sem substituir os valores originais da Silver.

- Leituras com `id_uc`, competência ou consumo ausentes, consumo negativo ou flag explícita de invalidade **não participam da comparação**.
- Para `id_uc + competência`, **duplicatas válidas geram erro** e devem ser corrigidas na Silver. Não usar `avg()` para esconder múltiplos registros.
- A flag `flag_consumo_inconsistente` cria um alerta de revisão; **não** presume fraude ou impede que o valor seja contabilizado na auditoria.
- A detecção de leitura estimada combina `flag_leitura_estimada` (nome real encontrado na Silver), alternativas legadas e `tipo_leitura`.
- Eventos de GD, medidores e inspeções só são considerados até o **fechamento da competência de corte**. O catálogo Silver do cadastro é uma fotografia atual, não um histórico temporal completo.

As exclusões de leitura e os critérios de qualidade serão exibidos na seção de QA. Nenhum dado-fonte é regravado.


In [0]:
def exigir(df, colunas, nome):
    faltantes = sorted(set(colunas) - set(df.columns))
    if faltantes:
        raise ValueError(f"{nome}: colunas da Silver ausentes: {faltantes}")

def flag(df, nome):
    if nome in df.columns:
        return F.coalesce(F.col(nome).cast("boolean"), F.lit(False))
    return F.lit(False)

exigir(uc, ["id_uc", "municipio", "bairro", "classe", "alimentador"], "cadastro")
exigir(leituras, ["id_uc", "mes_referencia", "consumo_kwh", "tipo_leitura"], "leituras")
exigir(medidores, ["id_uc", "data_retirada"], "medidores")
exigir(gd, ["id_uc", "data_conexao"], "geracao_distribuida")
exigir(inspecoes, ["id_inspecao", "id_uc", "data", "resultado", "kwh_recuperado"], "inspecoes")

assert uc.where(F.col("id_uc").isNull()).limit(1).count() == 0, "UC nula no cadastro"
assert uc.groupBy("id_uc").count().where(F.col("count") > 1).limit(1).count() == 0, "UC duplicada no cadastro"

flags_disponiveis = [c for c in ["flag_estimada", "flag_leitura_estimada", "flag_outlier",
                                  "flag_leitura_invalida", "flag_consumo_invalido",
                                  "flag_primeira_leitura_pos_troca", "flag_consumo_inconsistente"]
                      if c in leituras.columns]
leitura_mensal = (leituras
    .select("id_uc", "mes_referencia", "consumo_kwh", "tipo_leitura", *flags_disponiveis)
    .withColumn("mes", F.to_date(F.concat(
        F.substring(F.col("mes_referencia").cast("string"), 1, 7), F.lit("-01"))))
    .withColumn("consumo", F.col("consumo_kwh").cast("double"))
    .withColumn("estimada", flag(leituras, "flag_estimada") |
                flag(leituras, "flag_leitura_estimada") |
                (F.upper(F.trim(F.col("tipo_leitura").cast("string"))) == "ESTIMADA"))
    .withColumn("estimada", F.coalesce(F.col("estimada"), F.lit(False)))
    .withColumn("outlier", flag(leituras, "flag_outlier"))
    .withColumn("pos_troca", flag(leituras, "flag_primeira_leitura_pos_troca"))
    .withColumn("invalida", flag(leituras, "flag_leitura_invalida") |
                flag(leituras, "flag_consumo_invalido"))
    .withColumn("consumo_revisao", flag(leituras, "flag_consumo_inconsistente"))
)

condicao_invalida = (
    F.col("id_uc").isNull() | F.col("mes").isNull() |
    F.col("consumo").isNull() | (F.col("consumo") < 0) | F.col("invalida")
)
qtd_excluidas = leitura_mensal.where(condicao_invalida).count()
mensal = (leitura_mensal.where(~condicao_invalida)
    .select("id_uc", "mes", "consumo", "estimada", "outlier", "pos_troca", "consumo_revisao"))

repeticoes = mensal.groupBy("id_uc", "mes").count().where(F.col("count") > 1)
if repeticoes.limit(1).count():
    display(repeticoes.orderBy(F.desc("count")).limit(10))
    raise ValueError("Leituras Silver com UC+mês duplicados: corrigir na Silver, não aplicar avg() na Gold")

referencia = mensal.agg(F.max("mes").alias("mes_max")).first()["mes_max"]
assert referencia is not None, "Nenhuma competência válida na Silver de leituras"
print(f"Referência: {referencia} | leituras inválidas fora da análise: {qtd_excluidas}")

# Datas já tipadas na Silver. Não levar dados futuros para o score retrospectivo.
gd_datas = gd.select("id_uc", F.col("data_conexao").cast("date").alias("data_conexao"))
medidores_datas = medidores.select("id_uc", F.col("data_retirada").cast("date").alias("data_retirada"))
insp_base = inspecoes.select(
    "id_inspecao", "id_uc", F.col("data").cast("date").alias("data_inspecao"),
    F.upper(F.trim(F.col("resultado"))).alias("resultado"),
    F.col("kwh_recuperado").cast("double").alias("kwh_recuperado"))


## 4. Metodologia da fila de perdas

### 4.1. Consumo e comparação sazonal

Para cada UC, parear cada mês dos **3 meses de referência** ao mesmo mês do ano anterior, considerando apenas pares em que ambos os meses possuem consumo válido. A diferença mensal média positiva é:

\[\Delta kWh_{mês} = \max(0,\overline{kWh}_{ano\ anterior} - \overline{kWh}_{atual})\]

A queda relativa só é calculada quando a média anterior é positiva. O potencial bruto é a diferença mensal multiplicada pela **quantidade de meses efetivamente pareados**, não automaticamente por três.

### 4.2. Score e evidências

| Critério | Tratamento |
| --- | --- |
| Queda percentual de consumo | Até 55 pontos |
| Queda absoluta média | Até 30 pontos |
| Pelo menos dois **pares comparáveis** com leitura estimada | +10 pontos |
| GD conectada após o início da janela-base | −35 pontos e retenção para revisão |
| Troca recente ou leitura sinalizada pós-troca | −30 pontos e retenção para revisão |
| Inspeção nos 90 dias anteriores ao corte | −25 pontos e retenção para revisão |

Os pontos de queda dependem da existência de pares válidos. As leituras estimadas servem como **sinal de auditoria**, não como prova de perda. As evidências são classificadas como `ALTA`, `MEDIA`, `BAIXA` ou `INDETERMINADA` conforme a quantidade de pares medidos, estimados e com flags de qualidade.

### 4.3. Decisão de triagem

O campo `status_triagem` separa `ELEGIVEL`, `HISTORICO_INSUFICIENTE`, `SEM_EVIDENCIA_MINIMA`, `REVISAR_EVIDENCIA_ESTIMADA`, `REVISAR_FATORES_ALTERNATIVOS`, `REVISAR_INSPECAO_RECENTE` e `BAIXA_PRIORIDADE`. Apenas `ELEGIVEL` entra na fila operacional; os demais permanecem na auditoria, com `motivos`, `fatores_atenuantes` e métricas de suporte.

> **Limitações:** queda de consumo pode decorrer de sazonalidade residual, ocupação, mudança de atividade, eficiência energética, medidores ou GD. Um consumo alto em kWh pode dominar o ranking por oportunidade sem representar risco maior de irregularidade. Os pesos são hipóteses de triagem e precisam de validação futura.


In [0]:
def calcular_gold(mes_ref):
    """Calcula ranking sem eventos posteriores ao último dia da competência de corte."""
    ref = F.lit(mes_ref).cast("date")
    fim = F.last_day(ref)
    base_inicio = F.add_months(ref, -(12 + JANELA_MESES - 1))

    atual = mensal.where((F.col("mes") >= F.add_months(ref, -(JANELA_MESES - 1))) &
                         (F.col("mes") <= ref))
    ano_passado = (mensal
        .where((F.col("mes") >= base_inicio) & (F.col("mes") <= F.add_months(ref, -12)))
        .select("id_uc", F.add_months(F.col("mes"), 12).alias("mes"),
                F.col("consumo").alias("consumo_base"),
                F.col("estimada").alias("estimada_base"),
                F.col("outlier").alias("outlier_base"),
                F.col("consumo_revisao").alias("consumo_revisao_base")))

    pares = (atual.join(ano_passado, ["id_uc", "mes"], "left")
        .withColumn("par_valido", F.col("consumo_base").isNotNull())
        .withColumn("par_estimado", F.col("par_valido") &
                    (F.col("estimada") | F.coalesce(F.col("estimada_base"), F.lit(False))))
        .withColumn("par_observado", F.col("par_valido") & ~F.col("par_estimado"))
        .withColumn("par_inconsistente", F.col("par_valido") &
                    (F.col("consumo_revisao") |
                     F.coalesce(F.col("consumo_revisao_base"), F.lit(False)))))

    resumo = pares.groupBy("id_uc").agg(
        F.sum(F.col("par_valido").cast("int")).alias("meses_comparaveis"),
        F.count("*").alias("meses_atuais"),
        F.avg(F.when(F.col("par_valido"), F.col("consumo"))).alias("media_atual"),
        F.avg("consumo_base").alias("media_anterior"),
        F.sum(F.col("estimada").cast("int")).alias("meses_estimados"),
        F.sum(F.col("par_estimado").cast("int")).alias("pares_estimados"),
        F.sum(F.col("par_observado").cast("int")).alias("pares_observados"),
        F.max(F.col("par_inconsistente").cast("int")).alias("tem_inconsistencia_pareada"),
        F.max(F.when(F.col("par_valido"),
                     (F.col("outlier") | F.coalesce(F.col("outlier_base"), F.lit(False))))
              .otherwise(F.lit(False)).cast("int")).alias("tem_outlier_pareado"),
        F.max(F.col("pos_troca").cast("int")).alias("troca_sinalizada_silver"))

    # Na data de corte, considerar apenas eventos conhecidos até aquele momento.
    gd_uc = (gd_datas.where(F.col("data_conexao").isNotNull() & (F.col("data_conexao") <= fim))
        .groupBy("id_uc").agg(F.min("data_conexao").alias("primeira_conexao_gd")))
    gd_data_desconhecida = (gd_datas.where(F.col("data_conexao").isNull() &
                                     F.lit(mes_ref == referencia))
        .select("id_uc").distinct().withColumn("gd_data_desconhecida", F.lit(True)))
    trocas_uc = (medidores_datas
        .where(F.col("data_retirada").between(F.add_months(ref, -MESES_TROCA_RECENTE), fim))
        .groupBy("id_uc").agg(F.count("*").alias("trocas_recentes")))
    insp_passadas = (insp_base
        .where(F.col("data_inspecao").isNotNull() & (F.col("data_inspecao") <= fim))
        .groupBy("id_uc").agg(F.max("data_inspecao").alias("ultima_inspecao")))

    universo = (uc.select("id_uc", "municipio", "bairro", "classe", "alimentador")
        .join(resumo, "id_uc", "left")
        .join(gd_uc, "id_uc", "left")
        .join(gd_data_desconhecida, "id_uc", "left")
        .join(trocas_uc, "id_uc", "left")
        .join(insp_passadas, "id_uc", "left"))

    universo = (universo
        .fillna(0, subset=["meses_comparaveis", "meses_atuais", "meses_estimados",
                           "pares_estimados", "pares_observados"])
        .withColumn("historico_suficiente", F.col("meses_comparaveis") >= MIN_PARES)
        .withColumn("tem_outlier_pareado", F.coalesce(F.col("tem_outlier_pareado") > 0, F.lit(False)))
        .withColumn("tem_inconsistencia_pareada", F.coalesce(F.col("tem_inconsistencia_pareada") > 0, F.lit(False)))
        .withColumn("proporcao_pares_estimados", F.when(F.col("meses_comparaveis") > 0,
            F.col("pares_estimados") / F.col("meses_comparaveis")).otherwise(F.lit(None).cast("double")))
        .withColumn("confianca_evidencia", F.when(~F.col("historico_suficiente"), "INDETERMINADA")
            .when(F.col("pares_observados") < MIN_PARES_OBSERVADOS, "BAIXA")
            .when((F.col("proporcao_pares_estimados") >= LIMIAR_PROP_PARES_ESTIMADOS) |
                  F.col("tem_outlier_pareado") | F.col("tem_inconsistencia_pareada"), "MEDIA")
            .otherwise("ALTA"))
        .withColumn("gd_data_desconhecida", F.coalesce(F.col("gd_data_desconhecida"), F.lit(False)))
        .withColumn("possui_gd", F.col("primeira_conexao_gd").isNotNull() | F.col("gd_data_desconhecida"))
        .withColumn("gd_apos_base", F.coalesce(F.col("primeira_conexao_gd").between(base_inicio, fim), F.lit(False)))
        .withColumn("houve_troca_na_janela", F.coalesce(F.col("trocas_recentes") > 0, F.lit(False)) |
                    F.coalesce(F.col("troca_sinalizada_silver") > 0, F.lit(False)))
        .withColumn("inspecao_recente", F.coalesce(F.col("ultima_inspecao") >=
                    F.date_sub(fim, DIAS_INSPECAO_RECENTE - 1), F.lit(False))))

    universo = (universo
        .withColumn("queda_kwh_mensal", F.when(F.col("historico_suficiente") &
             (F.col("media_anterior") > 0),
             F.greatest(F.lit(0.0), F.col("media_anterior") - F.col("media_atual")))
             .otherwise(F.lit(0.0)))
        .withColumn("queda_pct", F.when(F.col("historico_suficiente") &
             (F.col("media_anterior") > 0),
             F.greatest(F.lit(0.0), (F.col("media_anterior") - F.col("media_atual")) /
                        F.col("media_anterior"))).otherwise(F.lit(0.0))))

    universo = (universo
        .withColumn("p_queda_pct", F.when(F.col("historico_suficiente"),
            F.least(F.lit(PESO_QUEDA_PCT_MAX), F.col("queda_pct") * 100.0))
            .otherwise(F.lit(0.0)))
        .withColumn("p_queda_kwh", F.when(F.col("historico_suficiente"),
            F.least(F.lit(PESO_QUEDA_KWH_MAX),
                    F.col("queda_kwh_mensal") * (PESO_QUEDA_KWH_MAX / 300.0)))
            .otherwise(F.lit(0.0)))
        .withColumn("p_estimada", F.when(F.col("historico_suficiente") &
            (F.col("pares_estimados") >= 2), F.lit(PESO_ESTIMADAS)).otherwise(F.lit(0.0)))
        .withColumn("score_bruto", F.col("p_queda_pct") + F.col("p_queda_kwh") + F.col("p_estimada"))
        .withColumn("penalidades",
            F.when(F.col("gd_apos_base"), F.lit(PENALIDADE_GD_APOS_BASE)).otherwise(F.lit(0.0)) +
            F.when(F.col("houve_troca_na_janela"), F.lit(PENALIDADE_TROCA)).otherwise(F.lit(0.0)) +
            F.when(F.col("inspecao_recente"), F.lit(PENALIDADE_INSPECAO)).otherwise(F.lit(0.0)))
        .withColumn("score", F.round(F.greatest(F.lit(0.0),
            F.least(F.lit(100.0), F.col("score_bruto") - F.col("penalidades"))), 2)))

    universo = (universo
        .withColumn("motivos", F.array(
            F.when((F.col("queda_pct") >= LIMIAR_QUEDA_PCT) &
                   (F.col("queda_kwh_mensal") >= LIMIAR_QUEDA_KWH_MES),
                   F.lit("QUEDA_SAZONAL_RELEVANTE")),
            F.when(F.col("queda_kwh_mensal") >= LIMIAR_ABS_ELEVADO_KWH_MES,
                   F.lit("QUEDA_ABSOLUTA_RELEVANTE")),
            F.when(F.col("historico_suficiente") & (F.col("pares_estimados") >= 2),
                   F.lit("LEITURAS_ESTIMADAS_RECORRENTES"))))
        .withColumn("motivos", F.expr("filter(motivos, x -> x is not null)"))
        .withColumn("fatores_atenuantes", F.array(
            F.when(F.col("gd_apos_base"), F.lit("GD_CONECTADA_APOS_BASE")),
            F.when(F.col("gd_data_desconhecida"), F.lit("GD_SEM_DATA_CONEXAO")),
            F.when(F.col("houve_troca_na_janela"), F.lit("TROCA_MEDIDOR_RECENTE")),
            F.when(F.col("inspecao_recente"), F.lit("INSPECAO_RECENTE")),
            F.when(F.col("tem_outlier_pareado"), F.lit("OUTLIER_NO_PAREAMENTO")),
            F.when(F.col("tem_inconsistencia_pareada"), F.lit("CONSUMO_COM_FLAG_INCONSISTENCIA")),
            F.when(F.col("pares_observados") < MIN_PARES_OBSERVADOS,
                   F.lit("AUSENCIA_PARES_MEDIDOS"))))
        .withColumn("fatores_atenuantes", F.expr("filter(fatores_atenuantes, x -> x is not null)")))

    universo = (universo
        .withColumn("status_triagem",
            F.when(~F.col("historico_suficiente"), F.lit("HISTORICO_INSUFICIENTE"))
             .when(F.size("motivos") == 0, F.lit("SEM_EVIDENCIA_MINIMA"))
             .when(F.col("pares_observados") < MIN_PARES_OBSERVADOS,
                   F.lit("REVISAR_EVIDENCIA_ESTIMADA"))
             .when(F.col("gd_apos_base") | F.col("gd_data_desconhecida") |
                   F.col("houve_troca_na_janela") | F.col("tem_outlier_pareado") |
                   F.col("tem_inconsistencia_pareada"),
                   F.lit("REVISAR_FATORES_ALTERNATIVOS"))
             .when(F.col("inspecao_recente"), F.lit("REVISAR_INSPECAO_RECENTE"))
             .when(F.col("score") < MIN_SCORE_ELEGIVEL, F.lit("BAIXA_PRIORIDADE"))
             .otherwise(F.lit("ELEGIVEL")))
        .withColumn("kwh_potencial_bruto", F.round(
            F.col("queda_kwh_mensal") * F.col("meses_comparaveis"), 2))
        .withColumn("kwh_estimado", F.when(F.col("status_triagem") == "ELEGIVEL",
            F.col("kwh_potencial_bruto")).otherwise(F.lit(0.0)))
        .withColumn("mes_referencia", ref)
        .withColumn("versao_regra", F.lit(VERSAO_REGRA))
        .withColumn("id_execucao", F.lit(ID_EXECUCAO))
        .withColumn("metodo_estimativa", F.lit("DIFERENCA_MEDIA_SAZONAL_X_MESES_PAREADOS")))

    colunas = ["id_uc", "municipio", "bairro", "classe", "alimentador", "score",
               "score_bruto", "penalidades", "motivos", "fatores_atenuantes",
               "kwh_estimado", "kwh_potencial_bruto", "queda_pct", "queda_kwh_mensal",
               "media_anterior", "media_atual", "meses_atuais", "meses_comparaveis",
               "meses_estimados", "pares_estimados", "pares_observados",
               "proporcao_pares_estimados", "confianca_evidencia", "historico_suficiente",
               "possui_gd", "primeira_conexao_gd", "gd_apos_base", "gd_data_desconhecida",
               "houve_troca_na_janela", "tem_outlier_pareado", "tem_inconsistencia_pareada",
               "ultima_inspecao", "inspecao_recente", "status_triagem", "mes_referencia",
               "versao_regra", "id_execucao", "metodo_estimativa"]
    return (universo.select(*colunas)
        .withColumn("kwh_estimado", F.col("kwh_estimado").cast("decimal(14,2)"))
        .withColumn("kwh_potencial_bruto", F.col("kwh_potencial_bruto").cast("decimal(14,2)")))

universo_atual = calcular_gold(referencia)
fila = universo_atual.where(F.col("status_triagem") == "ELEGIVEL")
print("Universo e fila calculados; ainda não publicados na Gold.")


## 5. Garantias de qualidade e revisão analítica

As verificações seguintes são **bloqueantes** para publicação: universo completo, chave `id_uc` única, inexistência de UCs órfãs, contagens de pares consistentes, score dentro de 0–100, proxies de kWh não negativos e nenhuma UC indevida na fila.

Também são exibidos, apenas durante a execução:

- Contagens por `status_triagem` e `confianca_evidencia`;
- Distribuição do score e do proxy de kWh por faixa;
- Composição por classe de consumo e distribuição geográfica, para detectar concentração operacional;
- Top 20 por score **versus** Top 20 por kWh, destacando que os dois objetivos não são equivalentes.

**Critério de aceite técnico:** todas as asserções devem passar **antes** da gravação. Caso contrário, investigar na Silver e reexecutar o pipeline; não mascarar inconsistências diretamente na Gold.


In [0]:
total_ucs = uc.count()
assert universo_atual.count() == total_ucs, "Cardinalidade incorreta: joins replicaram ou descartaram UCs"
assert universo_atual.groupBy("id_uc").count().where(F.col("count") > 1).limit(1).count() == 0, "UC duplicada na Gold"
assert universo_atual.join(uc.select("id_uc"), "id_uc", "left_anti").limit(1).count() == 0, "UC órfã na Gold"
assert universo_atual.where(
    F.col("score").isNull() | (F.col("score") < 0) | (F.col("score") > 100) |
    F.col("kwh_estimado").isNull() | (F.col("kwh_estimado") < 0) |
    F.col("kwh_potencial_bruto").isNull() | (F.col("kwh_potencial_bruto") < 0) |
    F.col("status_triagem").isNull() | F.col("confianca_evidencia").isNull() |
    (F.col("meses_comparaveis") > JANELA_MESES) |
    (F.col("pares_estimados") < 0) | (F.col("pares_observados") < 0) |
    ((F.col("kwh_estimado") > 0) & (F.col("status_triagem") != "ELEGIVEL")) |
    (F.col("pares_observados") + F.col("pares_estimados") != F.col("meses_comparaveis"))
).limit(1).count() == 0, "Score, evidência, contagens ou kWh inválidos"
assert fila.where((F.size("motivos") == 0) | (~F.col("historico_suficiente")) |
    (F.col("pares_observados") < MIN_PARES_OBSERVADOS) |
    (F.col("score") < MIN_SCORE_ELEGIVEL) | F.col("gd_apos_base") |
    F.col("gd_data_desconhecida") | F.col("houve_troca_na_janela") |
    F.col("inspecao_recente") | F.col("tem_outlier_pareado") |
    F.col("tem_inconsistencia_pareada")).limit(1).count() == 0, "Fila contém UC inelegível"

qtd_candidatos = fila.count()
print(f"UCs cadastro={total_ucs} | elegíveis={qtd_candidatos} | referência={referencia}")
print("Status e nível de confiança:")
display(universo_atual.groupBy("status_triagem", "confianca_evidencia")
    .agg(F.count("*").alias("ucs"), F.round(F.avg("score"), 2).alias("score_medio"))
    .orderBy("status_triagem", "confianca_evidencia"))

print("Scores, potencial energético e qualidade da evidência (elegíveis):")
faixas = (fila.withColumn("faixa_score", F.when(F.col("score") >= 80, "80–100")
    .when(F.col("score") >= 60, "60–79")
    .when(F.col("score") >= 40, "40–59").otherwise("20–39")))
display(faixas.groupBy("faixa_score").agg(
    F.count("*").alias("ucs"), F.round(F.sum("kwh_estimado"), 2).alias("kwh_proxy_total"),
    F.round(F.avg("pares_observados"), 2).alias("media_pares_observados"),
    F.round(F.avg("proporcao_pares_estimados") * 100, 1).alias("pct_pares_estimados"))
    .orderBy("faixa_score"))

print("Composição dos elegíveis por classe (verificar concentração):")
display(fila.groupBy("classe").agg(
    F.count("*").alias("ucs"), F.round(F.avg("score"), 2).alias("score_medio"),
    F.round(F.sum("kwh_estimado"), 2).alias("kwh_proxy_total"))
    .orderBy(F.desc("ucs")))
print("Maiores concentrações por município e alimentador:")
display(fila.groupBy("municipio", "alimentador").agg(
    F.count("*").alias("ucs"), F.round(F.sum("kwh_estimado"), 2).alias("kwh_proxy_total"))
    .orderBy(F.desc("ucs"), F.desc("kwh_proxy_total")).limit(15))
print("TOP 20 por score (risco heurístico):")
display(fila.orderBy(F.desc("score"), F.desc("kwh_estimado"), F.asc("id_uc"))
    .select("id_uc", "municipio", "bairro", "classe", "score", "confianca_evidencia",
            "motivos", "kwh_estimado", "pares_observados", "pares_estimados").limit(20))
print("TOP 20 por kWh potencial (oportunidade, NÃO receita garantida):")
display(fila.orderBy(F.desc("kwh_estimado"), F.desc("score"), F.asc("id_uc"))
    .select("id_uc", "municipio", "classe", "score", "confianca_evidencia",
            "kwh_estimado", "queda_pct").limit(20))


## 6. Avaliação retrospectiva (backtest **opcional**)

Com `executar_backtest=true`, o notebook cria **cortes históricos trimestrais** com janela de observação posterior definida de antemão. Para cada corte:

1. Calcula a fila usando apenas leituras/eventos com data até o fechamento do mês de corte;
2. Identifica a **primeira inspeção conclusiva** de cada UC nos três meses posteriores;
3. Compara **Top 50 e Top 200** com os demais elegíveis efetivamente inspecionados no mesmo período;
4. Reporta cobertura de rótulos, taxa de irregularidade observada e kWh recuperados registrados.

**Proteção contra interpretações incorretas:** quando não houver inspecionados suficientes fora do Top, o comparador e o *lift* serão marcados como **indisponíveis**, em vez de sugerir melhora não demonstrada. O conjunto inspecionado **não é aleatório**: as taxas não representam desempenho na população, recall, precisão real ou economia garantida. O cadastro da Silver é uma fotografia atual e pode introduzir viés de sobrevivência nos cortes passados.

O backtest é diagnóstico e não ajusta automaticamente os pesos. Os resultados devem ser confrontados com as regras operacionais de inspeção e revisados com a equipe de perdas.


In [0]:
# Backtest exploratório: cortes cronológicos e primeira inspeção conclusiva após o corte.
# Não há amostragem aleatória; métricas observacionais, não precisão populacional.
if EXECUTAR_BACKTEST:
    from datetime import date
    from pyspark.sql import Window

    def somar_meses(mes, n):
        indice = mes.year * 12 + mes.month - 1 + n
        return date(indice // 12, indice % 12 + 1, 1)

    resumo_datas = insp_base.where(F.col("data_inspecao").isNotNull()).agg(
        F.min("data_inspecao").alias("primeira"),
        F.max("data_inspecao").alias("ultima")).first()
    primeira = resumo_datas["primeira"]
    ultima = resumo_datas["ultima"]
    primeira_leitura = mensal.agg(F.min("mes").alias("minimo")).first()["minimo"]
    if primeira is None or primeira_leitura is None:
        print("Backtest indisponível: não há inspeções ou leituras datadas.")
    else:
        # Corte 1: mês anterior à primeira inspeção. Janela de avaliação: próximos 3 meses completos.
        corte_inicial = somar_meses(primeira.replace(day=1), -1)
        ultimo_corte = somar_meses(ultima.replace(day=1), -HORIZONTE_BACKTEST_MESES)
        cortes = []
        corte = corte_inicial
        while corte <= ultimo_corte and len(cortes) < MAX_CORTES_BACKTEST:
            meses_de_historico = (corte.year - primeira_leitura.year) * 12 + corte.month - primeira_leitura.month
            if meses_de_historico >= 12 + JANELA_MESES - 1:
                cortes.append(corte)
            corte = somar_meses(corte, PASSO_BACKTEST_MESES)

        if not cortes:
            print("Backtest indisponível: não há cortes com histórico suficiente e janela posterior completa.")
        else:
            metricas = []
            for ref_bt in cortes:
                corte_fim = F.last_day(F.lit(ref_bt).cast("date"))
                prox_inicio = F.date_add(corte_fim, 1)
                prox_fim = F.last_day(F.add_months(corte_fim, HORIZONTE_BACKTEST_MESES))
                # Uma UC pode ter múltiplas inspeções; usamos somente a primeira conclusiva.
                w_inspecao = Window.partitionBy("id_uc").orderBy(F.asc("data_inspecao"), F.asc("id_inspecao"))
                rotulos = (insp_base
                    .where(F.col("data_inspecao").between(prox_inicio, prox_fim))
                    .where(F.col("resultado").isin("IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE"))
                    .withColumn("ordem_inspecao", F.row_number().over(w_inspecao))
                    .where(F.col("ordem_inspecao") == 1)
                    .select("id_uc",
                        F.when(F.col("resultado") == "IRREGULARIDADE CONFIRMADA", 1).otherwise(0).alias("confirmada"),
                        F.when(F.col("resultado") == "IRREGULARIDADE CONFIRMADA",
                            F.greatest(F.lit(0.0), F.coalesce(F.col("kwh_recuperado"), F.lit(0.0))))
                            .otherwise(F.lit(0.0)).alias("kwh_recuperado_observado")))
                candidatos_bt = calcular_gold(ref_bt).where(F.col("status_triagem") == "ELEGIVEL")
                w_ranking = Window.orderBy(F.desc("score"), F.desc("kwh_estimado"), F.asc("id_uc"))
                classificados = candidatos_bt.withColumn("posicao", F.row_number().over(w_ranking))
                n_elegiveis = classificados.count()
                rotulados = classificados.join(rotulos, "id_uc", "inner")
                n_inspecionados = rotulados.count()
                for top_n in TOPS_BACKTEST:
                    n_top = min(top_n, n_elegiveis)
                    resultado = rotulados.agg(
                        F.sum(F.when(F.col("posicao") <= n_top, 1).otherwise(0)).alias("rotulos_top"),
                        F.sum(F.when((F.col("posicao") <= n_top) & (F.col("confirmada") == 1), 1).otherwise(0)).alias("confirmadas_top"),
                        F.sum(F.when(F.col("posicao") > n_top, 1).otherwise(0)).alias("rotulos_outros"),
                        F.sum(F.when((F.col("posicao") > n_top) & (F.col("confirmada") == 1), 1).otherwise(0)).alias("confirmadas_outros"),
                        F.sum(F.when(F.col("posicao") <= n_top, F.col("kwh_recuperado_observado"))
                            .otherwise(F.lit(0.0))).alias("kwh_recuperado_top"),
                    ).first()
                    rtop = int(resultado["rotulos_top"] or 0)
                    ctop = int(resultado["confirmadas_top"] or 0)
                    routros = int(resultado["rotulos_outros"] or 0)
                    coutros = int(resultado["confirmadas_outros"] or 0)
                    taxa_top = ctop / rtop if rtop else None
                    taxa_outros = coutros / routros if routros else None
                    lift = taxa_top / taxa_outros if (taxa_top is not None and taxa_outros is not None and taxa_outros > 0) else None
                    if routros == 0 or rtop == 0:
                        status = "SEM_COMPARADOR_OBSERVADO"
                        lift = None
                    elif min(rtop, routros) < MIN_ROTULOS_COMPARACAO:
                        status = "AMOSTRA_OBSERVADA_PEQUENA"
                    else:
                        status = "COMPARACAO_OBSERVACIONAL"
                    metricas.append((str(ref_bt), int(top_n), n_elegiveis, int(n_top),
                        n_inspecionados, rtop, ctop, routros, coutros, taxa_top,
                        taxa_outros, lift, rtop/n_top if n_top else None,
                        float(resultado["kwh_recuperado_top"] or 0.0), status))
                print(f"Corte {ref_bt}: elegíveis={n_elegiveis}, com inspeção observada={n_inspecionados}"
                      f" nos {HORIZONTE_BACKTEST_MESES} meses seguintes.")
            # Schema explícito evita inferência incorreta quando comparadores não têm rótulos.
            colunas_bt = ["competencia_corte", "top_planejado", "elegiveis_total", "top_real",
                "elegiveis_inspecionados", "top_inspecionados", "confirmadas_top",
                "outros_inspecionados", "confirmadas_outros", "taxa_top", "taxa_outros",
                "razao_taxas", "cobertura_rotulos_top", "kwh_recuperado_observado_top", "qualidade_comparacao"]
            tipos_bt = "string, int, long, int, long, long, long, long, long, double, double, double, double, double, string"
            schema_bt = ", ".join(f"{col} {tipo}" for col,tipo in zip(colunas_bt,tipos_bt.split(", ")))
            display(spark.createDataFrame(metricas, schema=schema_bt).orderBy("competencia_corte", "top_planejado"))
            print("ALERTA: resultados condicionados à seleção das UCs que foram inspecionadas;")
            print("não extrapolar taxas ou kWh observados para toda a população.")
else:
    print("Backtest desligado para reduzir consumo do Serverless. Use executar_backtest=true para avaliação.")


## 7. Publicação Delta e rastreabilidade

**Somente após passar no QA**, o notebook sobrescreve:

- `gold.tb_fila_perdas_auditoria`: universo completo e razões de elegibilidade/retenção;
- `gold.tb_fila_perdas`: apenas candidatos elegíveis.

As duas tabelas recebem `mes_referencia`, `versao_regra` e `id_execucao`. São gravadas individualmente em transações Delta; **não existe transação única entre as duas**. O notebook valida ID e cardinalidade depois de gravar, mas consumidores do portal devem evitar leituras durante atualização parcial.

As ordens de inspeção e seu histórico são **transacionais**, mantidos fora da Gold e preservados nas recargas. Não registrar ou atualizar ordens neste notebook.


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
nome_gold = f"{GOLD}.tb_fila_perdas"
nome_auditoria = f"{GOLD}.tb_fila_perdas_auditoria"

(universo_atual.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(nome_auditoria))
(fila.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(nome_gold))

# Checagem mínima após gravação: versões iguais, sem esperar alterações externas.
assert spark.table(nome_gold).where(
    F.col("id_execucao").isNull() | (F.col("id_execucao") != ID_EXECUCAO)).limit(1).count() == 0
assert spark.table(nome_auditoria).where(
    F.col("id_execucao").isNull() | (F.col("id_execucao") != ID_EXECUCAO)).limit(1).count() == 0
assert spark.table(nome_gold).count() == qtd_candidatos, "Quantidade publicada difere da validada"
assert spark.table(nome_auditoria).count() == total_ucs, "Universo da auditoria incompleto"
assert spark.table(nome_gold).where(F.col("versao_regra") != VERSAO_REGRA).limit(1).count() == 0
print(f"Gold atualizada: {nome_gold}")
print(f"Auditoria atualizada: {nome_auditoria}")
print(f"Regra={VERSAO_REGRA}; execução={ID_EXECUCAO}; referência={referencia}; elegíveis={qtd_candidatos}")


## 8. Integração com o portal e conclusão técnica

### Contrato de consumo sugerido

A rota `GET /inspecoes/candidatos` (implementada no backend) deve consultar **somente** `gold.tb_fila_perdas`, com ordenação estável por `score DESC`, `kwh_estimado DESC`, `id_uc ASC`, filtros e paginação. Antes de abrir uma ordem, o backend precisa verificar se já existe **ordem ativa** para aquela UC no banco transacional.

| Campo | Significado |
| --- | --- |
| `id_uc`, `municipio`, `bairro`, `classe`, `alimentador` | Identificação e filtros de localização |
| `score` | Pontuação heurística (0–100), não probabilidade |
| `motivos`, `fatores_atenuantes` | Explicabilidade e sinalização de revisão |
| `kwh_estimado` | Proxy de perda de consumo sazonal; não kWh recuperado |
| `confianca_evidencia`, `pares_observados`, `pares_estimados` | Qualidade da comparação |
| `mes_referencia`, `versao_regra`, `id_execucao` | Rastreabilidade da publicação |

**Saída esperada:** fila reproduzível para apoiar a triagem, com justificativa por UC, histórico de qualidade e separação dos casos duvidosos. A validação histórica é limitada a UCs que foram inspecionadas; não autoriza alegar eficácia causal ou recuperação financeira. A inspeção física permanece o meio de confirmação.

### Checklist de entrega

- [ ] Bronze e Silver das fontes executadas sem erro;
- [ ] QA e reconciliação do universo aprovados;
- [ ] Tabelas Gold gravadas com a **mesma** `id_execucao` e contagens coerentes;
- [ ] Contrato de colunas alinhado com `GET /inspecoes/candidatos`;
- [ ] Código versionado **sem outputs de execução nem dados confidenciais**;
- [ ] README principal referencia o notebook, sua sequência de execução e limitações.


In [0]:
print("TOP 20 publicado:")
display(spark.table(nome_gold)
    .orderBy(F.desc("score"), F.desc("kwh_estimado"), F.asc("id_uc"))
    .select("id_uc", "municipio", "bairro", "classe", "score", "confianca_evidencia",
            "motivos", "fatores_atenuantes", "kwh_estimado", "mes_referencia", "versao_regra")
    .limit(20))

print("Exemplos dos casos retidos para revisão (auditoria):")
display(spark.table(nome_auditoria)
    .where(F.col("status_triagem").startswith("REVISAR"))
    .select("id_uc", "status_triagem", "confianca_evidencia", "pares_observados",
            "pares_estimados", "fatores_atenuantes", "motivos", "score", "kwh_potencial_bruto")
    .limit(20))
